<a href="https://colab.research.google.com/github/amber-pan/Self-taught-AI-Engineer/blob/main/MSAI_ML_incremental_Capstone_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Created by: Amber Pan
**Movie recommendation:**

Study the various Recommendation Techniques for recommending movies using movies.csv, ratings.csv datasets

1. Load and merge ratings.csv and movies.csv.
2. Build a User-Item matrix.
3. User-Based Collaborative Filtering and predict User 1's rating for movieId 32.
4. Item-Based Collaborative Filtering and find 10 movies similar to Jurassic Park (1993).
5. KNNBasic model evaluation.
6. SVD model evaluation.
7. NMF model evaluation.
8. Compare RMSE scores across models.

In [ ]:
!pip install surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

from surprise import Dataset
from surprise import Reader
from surprise import KNNBasic
from surprise import SVD
from surprise import NMF
from surprise.model_selection import cross_validate

In [ ]:
# Load movies.csv and ratings.csv dataset
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

print(ratings.shape)
print(movies.shape)

ratings.head()

(100836, 4)
(9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# Merge both data frames on movieid

# Merge datasets
movie_data = pd.merge(
    ratings,
    movies,
    on="movieId",
    how="inner"
)

movie_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [ ]:
# create user-item matrix
# # Create User-Item Matrix (Hint: Use pandas pivot_table method with index = 'userId', columns = 'title', values = 'rating' )
user_item_matrix = movie_data.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

print(user_item_matrix.shape)

user_item_matrix.head()

(610, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# fill missing values with user mean
# Perform User-based Collaborative Filtering
# Fill the row-wise NaNs in the User-Item Matrix with the corresponding user's mean ratings, and find the Pearson correlation between users

user_mean_matrix = user_item_matrix.apply(
    lambda row: row.fillna(row.mean()),
    axis=1
)

user_mean_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,...,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.366379,4.000000,4.366379
2,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,...,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276,3.948276
3,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,...,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897,2.435897
4,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,...,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556,3.555556
5,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,...,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364,3.636364


In [ ]:
# use correlation matrix
# Choose the correlation of all users with only User 1
# Sort the User 1 correlation in the descending order

user_corr = user_mean_matrix.T.corr(method='pearson')

user_corr.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,1.264516e-03,5.525772e-04,0.048419,0.021847,-0.045497,-6.199672e-03,0.047013,1.950985e-02,-8.754088e-03,...,0.018127,-0.017172,-0.015221,-3.705875e-02,-2.912138e-02,0.012016,0.055261,0.075224,-2.571255e-02,0.010932
2,0.001265,1.000000e+00,-4.975816e-17,-0.017164,0.021796,-0.021051,-1.111357e-02,-0.048085,7.652805e-16,3.011629e-03,...,-0.050551,-0.031581,-0.001688,-1.265569e-15,-6.430371e-16,0.006226,-0.020504,-0.006001,-6.009100e-02,0.024999
3,0.000553,-4.975816e-17,1.000000e+00,-0.011260,-0.031539,0.004800,-2.587070e-16,-0.032471,-4.812885e-16,3.774064e-16,...,-0.004904,-0.016117,0.017749,-8.106536e-16,-1.430628e-03,-0.037289,-0.007789,-0.013001,-1.168878e-16,0.019550
4,0.048419,-1.716402e-02,-1.125978e-02,1.000000,-0.029620,0.013956,5.809139e-02,0.002065,-5.873603e-03,5.159032e-02,...,-0.037687,0.063122,0.027640,-1.378212e-02,4.003747e-02,0.020590,0.014628,-0.037569,-1.788358e-02,-0.000995
5,0.021847,2.179571e-02,-3.153892e-02,-0.029620,1.000000,0.009111,1.011715e-02,-0.012284,7.750436e-16,-3.316512e-02,...,0.015964,0.012427,0.027076,1.246135e-02,-3.627206e-02,0.026319,0.031896,-0.001751,9.382892e-02,-0.000278


In [ ]:
# use user 1 correlation
#. Drop the NaN values generated in the correlation matrix

user1_corr = user_corr[1]

user1_corr = user1_corr.sort_values(
    ascending=False
)

user1_corr = user1_corr.dropna()

user1_corr.head(10)

,1
userId,
1,1.000000
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070
369,0.098295
206,0.096852
535,0.096493


In [ ]:
# top 50 similar users
# Choose the top 50 users that are highly correlated to User 1

top50_users = user1_corr.iloc[1:51]

top50_users.head()

,1
userId,
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070


In [ ]:
# Obtain movie 32 title info
#
movie32_title = movies.loc[
    movies['movieId'] == 32,
    'title'
].values[0]

print(movie32_title)

Twelve Monkeys (a.k.a. 12 Monkeys) (1995)


In [ ]:
# gather rating for similar users
#Predict the rating that User 1 might give for the movie with movieid 32 based on the top 50 user correlation matrix
# (Hint: Predicted rating = sum of [(weights) * (ratings)] / sum of (weights ). Here, weights is the correlation of the corresponding user with the first user). That is, the predicted rating is calculated as the weighted average of k similar users
similar_users = top50_users.index

ratings_movie32 = user_item_matrix.loc[
    similar_users,
    movie32_title
]

prediction_df = pd.DataFrame({
    'weight': top50_users,
    'rating': ratings_movie32
})

prediction_df = prediction_df.dropna()

prediction_df.head()

,weight,rating
userId,,
414,0.101348,5.0
477,0.099217,4.5
57,0.099070,4.0
206,0.096852,3.0
590,0.095191,3.0


In [ ]:
# weighted average
predicted_rating = (
    np.sum(
        prediction_df['weight'] *
        prediction_df['rating']
    )
    /
    np.sum(prediction_df['weight'])
)

print("Predicted Rating for User 1:", predicted_rating)

Predicted Rating for User 1: 4.13689530159743


In [ ]:
# check if the user 1 has rated movie 32 in original movie_data
selection_rating = movie_data.loc[
    (movie_data['userId'] == 1) &
    (movie_data['movieId'] == 32),
    'rating'
]

if not selection_rating.empty:
    original_rating = selection_rating.values[0]
    print(f"Original Rating for User 1 for movie {movie32_title}:", original_rating)
else:
    print(f"User 1 has not rated movie {movie32_title} (movieId 32) in the dataset.")


User 1 has not rated movie Twelve Monkeys (a.k.a. 12 Monkeys) (1995) (movieId 32) in the dataset.


In [ ]:
# check what user 1 has rated movie 32 in user_item_matrix
user_item_matrix.loc[
    1,
    movie32_title
]

np.float64(nan)

6. Item-Based Collaborative Filtering
Fill Missing Values with Movie Mean


In [ ]:
# fill missing value with movie mean
# Fill the column-wise NaN's in the User-Item Matrix with the corresponding movie's mean ratings, and find Pearson correlation between movies


item_mean_matrix = user_item_matrix.copy()

item_mean_matrix = item_mean_matrix.apply(
    lambda col: col.fillna(col.mean()),
    axis=0
)

item_mean_matrix.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,4.0,3.5,5.0,4.0,1.5,3.176471,3.0,3.666667,3.285714,...,1.5,4.0625,3.666667,3.0,3.0,3.863636,2.770833,2.0,4.000000,1.0
2,4.0,4.0,3.5,5.0,4.0,1.5,3.176471,3.0,3.666667,3.285714,...,1.5,4.0625,3.666667,3.0,3.0,3.863636,2.770833,2.0,3.134615,1.0
3,4.0,4.0,3.5,5.0,4.0,1.5,3.176471,3.0,3.666667,3.285714,...,1.5,4.0625,3.666667,3.0,3.0,3.863636,2.770833,2.0,3.134615,1.0
4,4.0,4.0,3.5,5.0,4.0,1.5,3.176471,3.0,3.666667,3.285714,...,1.5,4.0625,3.666667,3.0,3.0,3.863636,2.770833,2.0,3.134615,1.0
5,4.0,4.0,3.5,5.0,4.0,1.5,3.176471,3.0,3.666667,3.285714,...,1.5,4.0625,3.666667,3.0,3.0,3.863636,2.770833,2.0,3.134615,1.0


In [ ]:
# movie correlation matrix

movie_corr = item_mean_matrix.corr(
    method='pearson'
)

movie_corr.head()

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,1.0,NaN,-2.470228e-17,NaN,-6.783236e-16,-4.090781e-16,...,NaN,1.491400e-15,-2.970631e-18,NaN,NaN,-1.174172e-16,-4.643126e-16,-1.457468e-15,2.901144e-15,NaN


In [ ]:
# correlation with Jurassic Park 1993
# Choose the correlation of all movies with the movie Jurassic Park (1993) only
# Drop the NaN values generated in the correlation matrix
jurassic_corr = movie_corr[
    'Jurassic Park (1993)'
]

jurassic_corr = jurassic_corr.sort_values(
    ascending=False
)

jurassic_corr = jurassic_corr.dropna()

jurassic_corr.head(20)


,Jurassic Park (1993)
title,
Jurassic Park (1993),1.000000
"Fugitive, The (1993)",0.324717
Lethal Weapon (1987),0.318646
Independence Day (a.k.a. ID4) (1996),0.263629
Mission: Impossible (1996),0.258080
Ghostbusters (a.k.a. Ghost Busters) (1984),0.256527
Mulan (1998),0.255672
Rise of the Planet of the Apes (2011),0.248134
"Bug's Life, A (1998)",0.240964


In [ ]:
# top 10 similar movies
# Sort the  Jurassic Park movie correlation in descending order

# Find 10 movies similar to the movie Jurassic Park (1993)

similar_movies = jurassic_corr.iloc[1:11]

print("Top 10 Similar Movies\n")

for movie in similar_movies.index:
    print(movie)

Top 10 Similar Movies

Fugitive, The (1993)
Lethal Weapon (1987)
Independence Day (a.k.a. ID4) (1996)
Mission: Impossible (1996)
Ghostbusters (a.k.a. Ghost Busters) (1984)
Mulan (1998)
Rise of the Planet of the Apes (2011)
Bug's Life, A (1998)
Indiana Jones and the Temple of Doom (1984)
Die Hard (1988)


Perform KNNBasic, SVD, NMF Model-based Collaborative Filtering



In [ ]:

# prepare data for Surprise models
# Reader = metadata that tells Surprise how to interpret the ratings data
# Tell Surprise what a valid rating looks like.
# Initialize KNNBasic with similarity configuration as Mean Squared Distance Similarity (msd), 20 neighbors and cross-validate 5 folds against measure RMSE.
# (Hint: cross_validate(algo=algo, data=data, measures=['RMSE'], cv=5, verbose=True))

reader = Reader(
    rating_scale=(
        ratings.rating.min(),
        ratings.rating.max()
    )
)

data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']],
    reader
)

In [ ]:
# KNNBasic model with Mean Square Distance
sim_options = {
    'name': 'msd',
    'user_based': True
}

knn_model = KNNBasic(
    k=20,
    sim_options=sim_options
)

#cross validation
knn_results = cross_validate(
    algo=knn_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("KNN RMSE Mean:",
      np.mean(knn_results['test_rmse']))
# The cross_validate function, which was used, automatically performs cross-validation. This means it splits the data into multiple folds (in this case, 5 folds as indicated by cv=5). For each fold, it uses a portion of the data for training and the remaining portion as the test set. The test_rmse values that you see are the Root Mean Squared Error calculated on the test set for each of these 5 folds. Taking the np.mean() of these values gives you the average RMSE across all the test sets, providing a more robust estimate of the model's performance.

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9315  0.9490  0.9412  0.9345  0.9413  0.9395  0.0061  
Fit time          0.11    0.13    0.13    0.12    0.13    0.12    0.01    
Test time         1.04    1.16    1.04    1.14    1.07    1.09    0.05    
KNN RMSE Mean: 0.9395015912232635


In [ ]:
# SVD model
# Initialize Singular Value Decomposition (SVD) and  cross-validate 5 folds against measure RMSE.

svd_model = SVD()

svd_results = cross_validate(
    algo=svd_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("SVD RMSE Mean:",
      np.mean(svd_results['test_rmse']))

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8714  0.8764  0.8656  0.8727  0.8803  0.8733  0.0049  
Fit time          1.38    1.61    1.16    1.14    1.18    1.29    0.18    
Test time         0.18    0.23    0.11    0.21    0.13    0.17    0.05    
SVD RMSE Mean: 0.8732936348589513


In [ ]:
# NMF model
#NMF discovers hidden movie characteristics and hidden user preferences. It represents both users and movies using only positive latent factors, then predicts ratings by matching user preferences with movie characteristics.
# Initialize Non-Negative Matrix Factorization (NMF) and cross-validate 5 folds against measure RMSE.

nmf_model = NMF()

nmf_results = cross_validate(
    algo=nmf_model,
    data=data,
    measures=['RMSE'],
    cv=5,
    verbose=True
)

print("NMF RMSE Mean:",
      np.mean(nmf_results['test_rmse']))

Evaluating RMSE of algorithm NMF on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9235  0.9276  0.9214  0.9217  0.9249  0.9238  0.0023  
Fit time          2.38    2.51    3.10    2.41    2.41    2.56    0.27    
Test time         0.23    0.16    0.09    0.22    0.09    0.16    0.06    
NMF RMSE Mean: 0.9238466928418116


In [ ]:
# compare model performances
# Print best score and best params from Cross Validate on all the models built.
results = pd.DataFrame({
    'Model': ['KNNBasic', 'SVD', 'NMF'],
    'RMSE': [
        np.mean(knn_results['test_rmse']),
        np.mean(svd_results['test_rmse']),
        np.mean(nmf_results['test_rmse'])
    ]
})

results = results.sort_values(
    by='RMSE'
)

results

,Model,RMSE
1,SVD,0.873294
2,NMF,0.923847
0,KNNBasic,0.939502


	Model	RMSE
1	SVD	0.873294
2	NMF	0.923847
0	KNNBasic	0.939502


In [ ]:
#best model
best_model = results.iloc[0]

print(
    f"Best Model: {best_model['Model']}"
)

print(
    f"Best RMSE: {best_model['RMSE']:.4f}"
)

Best Model: SVD
Best RMSE: 0.8733


### **use grid search so that you can get best params**
only grid search can provide params so

In [ ]:
#  obrain best score and best params from Grid Search Cross Validate on all the models built.
from surprise.model_selection import GridSearchCV

In [ ]:
param_grid_knn = {
    'k': [10, 20, 30, 40],
    'sim_options': {
        'name': ['msd', 'cosine', 'pearson'],
        'user_based': [True, False]
    }
}

gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_knn.fit(data)

In [ ]:
print("KNN Best RMSE:")
print(gs_knn.best_score['rmse'])

print("\nKNN Best Parameters:")
print(gs_knn.best_params['rmse'])

KNN Best RMSE:
0.9084977985235406

KNN Best Parameters:
{'k': 40, 'sim_options': {'name': 'msd', 'user_based': False}}


In [29]:
param_grid_svd = {
    'n_factors': [50, 100, 150],
    'n_epochs': [20, 30],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.02, 0.1]
}

gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_svd.fit(data)

In [30]:
print("SVD Best RMSE:")
print(gs_svd.best_score['rmse'])

print("\nSVD Best Parameters:")
print(gs_svd.best_params['rmse'])

SVD Best RMSE:
0.8645485444100469

SVD Best Parameters:
{'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.1}


In [31]:
param_grid_nmf = {
    'n_factors': [15, 30, 50],
    'n_epochs': [50, 100],
    'reg_pu': [0.02, 0.06],
    'reg_qi': [0.02, 0.06]
}

gs_nmf = GridSearchCV(
    NMF,
    param_grid_nmf,
    measures=['rmse'],
    cv=5,
    n_jobs=-1
)

gs_nmf.fit(data)

In [ ]:
print("NMF Best RMSE:")
print(gs_nmf.best_score['rmse'])

print("\nNMF Best Parameters:")
print(gs_nmf.best_params['rmse'])

In [ ]:
results = pd.DataFrame({
    'Model': ['KNNBasic', 'SVD', 'NMF'],
    'Best RMSE': [
        gs_knn.best_score['rmse'],
        gs_svd.best_score['rmse'],
        gs_nmf.best_score['rmse']
    ]
})

results = results.sort_values(
    by='Best RMSE'
)

results

In [34]:
best_model = results.iloc[0]

print(f"Best Model: {best_model['Model']}")
print(f"Best RMSE: {best_model['Best RMSE']:.4f}")

Best Model: SVD
Best RMSE: 0.8645
